# Olist Delivery Delay Prediction
### Brazilian E-Commerce Public Dataset | Capstone Project
#### Notebook 02 : Data Merging

---

| Section | Details |
|--------|--------|
| **Input** | 9 cleaned CSV files from `data/processed/` (output of NB01) |
| **Output** | See details below |
| **Previous** | `01_data_understanding_cleaning.ipynb` |
| **Next** | `03_eda.ipynb` |

**Output Files:**

- `olist_merged.csv` (96,455 × 36)  
- `items_enriched.csv` (112,650 rows)  

Saved to `data/processed/`

---

## Data Integration Approach

This notebook combines all cleaned tables into **two output files**:

| File | Granularity | Purpose |
|------|-------------|---------|
| `olist_merged.csv` | One row = one order | EDA + Modeling base |
| `items_enriched.csv` | One row = one item | Seller & product deep-dives in EDA / NB04 |

### Core rules applied here

1. **No selection, only description** - when an order has multiple items or sellers, we never
   pick one representative. Instead we describe the whole picture with numeric aggregations
   (`sum`, `max`, `min`, `nunique`).

2. **No assumptions before EDA** - questions like *"which seller matters most?"* or
   *"which product category should represent the order?"* are answered by data in NB03 (EDA),
   not by gut-feel here.

3. **Seller geographic features are deferred** - `seller_lat/lng` depend on *which* seller we
   attach to the order. Since we do not choose a single seller here, those coordinates are
   computed in NB04 (Feature Engineering) after EDA identifies the relevant seller dimension
   (e.g. the seller with the latest `shipping_limit_date`, or the farthest from the customer).

4. **All columns are preserved** - EDA-only columns (`delivery_days`, `delivery_status`) and
   potential-leakage columns stay in the dataset. They are documented and excluded at the
   Feature Engineering stage, not here.

---

 **Next steps after this notebook:**
 - NB03 - EDA: use `olist_merged.csv` + `items_enriched.csv`
 - NB04 - Feature Engineering guided by EDA findings

---
## Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

---
## Load Cleaned Tables

All datasets were cleaned and saved in Notebook 01. We reload them here with proper date parsing.

In [2]:
PATH = '../data/processed/'

date_cols_orders = [
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

orders      = pd.read_csv(PATH + 'orders_clean.csv',      parse_dates=date_cols_orders)
items       = pd.read_csv(PATH + 'items_clean.csv',        parse_dates=['shipping_limit_date'])
payments    = pd.read_csv(PATH + 'payments_clean.csv')
reviews     = pd.read_csv(PATH + 'reviews_clean.csv')
customers   = pd.read_csv(PATH + 'customers_clean.csv')
sellers     = pd.read_csv(PATH + 'sellers_clean.csv')
products    = pd.read_csv(PATH + 'products_clean.csv')
geolocation = pd.read_csv(PATH + 'geolocation_clean.csv')
translation = pd.read_csv(PATH + 'translation_clean.csv')

tables = {
    'orders': orders, 'items': items, 'payments': payments,
    'reviews': reviews, 'customers': customers, 'sellers': sellers,
    'products': products, 'geolocation': geolocation, 'translation': translation
}

for name, df in tables.items():
    print(f'{name:<15} {df.shape[0]:>10,} rows | {df.shape[1]:>2} columns')

orders              96,455 rows | 13 columns
items              112,650 rows |  9 columns
payments           103,886 rows |  5 columns
reviews             99,224 rows |  7 columns
customers           99,441 rows |  6 columns
sellers              3,095 rows |  5 columns
products            32,949 rows |  9 columns
geolocation      1,000,163 rows |  7 columns
translation             71 rows |  2 columns


---
## Geolocation Aggregation

The geolocation table has ~1M rows - multiple coordinate records per zip code.

**Strategy:** aggregate to one centroid (mean lat/lng) per zip code prefix.

**Scope:** we build a `customer_geo` lookup only.
Seller coordinates are **not** attached here because we do not yet know which seller
to represent for multi-seller orders. That join happens in NB04 after EDA.

In [3]:
# ── Aggregate geolocation → one centroid per zip code ────────────────────────
geo_lookup = (
    geolocation
    .groupby('geolocation_zip_code_prefix', as_index=False)
    .agg(lat=('geolocation_lat', 'mean'), lng=('geolocation_lng', 'mean'))
)

customer_geo = geo_lookup.rename(columns={
    'geolocation_zip_code_prefix': 'customer_zip_code_prefix',
    'lat': 'customer_lat',
    'lng': 'customer_lng'
})

print(f'geo_lookup : {len(geo_lookup):,} unique zip code prefixes')
print(f'customer_geo: ready for left-join on customer_zip_code_prefix')
print()
print('Note: seller_geo is intentionally deferred to NB04.')
print('Reason: attaching seller coordinates requires choosing one seller')
print('        per order — a decision that depends on EDA findings.')

geo_lookup : 19,015 unique zip code prefixes
customer_geo: ready for left-join on customer_zip_code_prefix

Note: seller_geo is intentionally deferred to NB04.
Reason: attaching seller coordinates requires choosing one seller
        per order — a decision that depends on EDA findings.


---
## Fix Missing Category Translations

Two product categories exist in the products table but are absent from the translation table.
We detect them programmatically and add manual translations before the items join.

In [4]:
# ── Detect categories missing from translation table ─────────────────────────
missing_categories = (
    set(products['product_category_name'].dropna())
    - set(translation['product_category_name'])
)
missing_categories.discard('unknown')
print('Categories missing from translation table:', missing_categories)

Categories missing from translation table: {'portateis_cozinha_e_preparadores_de_alimentos', 'pc_gamer'}


In [5]:
# ── Patch translation table ──────────────────────────────────────────────────
manual_translations = {
    'pc_gamer'                                       : 'pc_gamer',
    'portateis_cozinha_e_preparadores_de_alimentos'  : 'portable_kitchen_food_preparators',
}

for pt, en in manual_translations.items():
    if pt not in translation['product_category_name'].values:
        translation = pd.concat([
            translation,
            pd.DataFrame([{'product_category_name': pt,
                           'product_category_name_english': en}])
        ], ignore_index=True)

print(f'Translation table: {len(translation)} categories (patched)')
print('All real categories now covered.')

Translation table: 73 categories (patched)
All real categories now covered.


---
## Build items_enriched

We enrich each item row with its product details, English category, and seller location.
This creates a **item-level** table where every row knows its full context.

This table serves two purposes:
- **Saved as `items_enriched.csv`** for seller and product analysis in EDA (NB03)
- **Used below** to aggregate order-level features without losing information

In [6]:
# ── Enrich each item with product info, category, and seller ─────────────────
items_enriched = (
    items
    .merge(products,    on='product_id',           how='left')
    .merge(translation, on='product_category_name', how='left')
    .merge(
        sellers[['seller_id', 'seller_zip_code_prefix',
                 'seller_state', 'seller_city']],
        on='seller_id', how='left'
    )
)

# Compute product volume
items_enriched['product_volume_cm3'] = (
    items_enriched['product_length_cm'] *
    items_enriched['product_height_cm'] *
    items_enriched['product_width_cm']
)

print(f'items_enriched: {len(items_enriched):,} rows x {items_enriched.shape[1]} columns')
print(f'Covers {items_enriched["order_id"].nunique():,} unique orders')
print()
print('Columns added vs raw items table:')
new_cols = [c for c in items_enriched.columns if c not in items.columns]
for c in new_cols:
    print(f'  + {c}')

items_enriched: 112,650 rows x 22 columns
Covers 98,666 unique orders

Columns added vs raw items table:
  + product_category_name
  + product_name_lenght
  + product_description_lenght
  + product_photos_qty
  + product_weight_g
  + product_length_cm
  + product_height_cm
  + product_width_cm
  + product_category_name_english
  + seller_zip_code_prefix
  + seller_state
  + seller_city
  + product_volume_cm3


---
## Aggregate Items to Order Level

**Design decisions explained:**

| What | How | Why |
|------|-----|-----|
| Item count | `max(order_item_id)` | Sequential item IDs - max = total items |
| Price | `sum` + `max` | Total cost + most expensive item |
| Freight | `sum` | Cumulative shipping cost |
| Weight | `sum` + `max` | Total load + heaviest single item |
| Volume | `sum` + `max` | Total bulk + largest single item |
| Category | `nunique` only | We do **not** pick one representative category - EDA decides |
| Sellers | `nunique` | Count only - we do **not** pick one seller - EDA decides |
| Seller states | `nunique` + `has_multi_state` | Measures geographic dispersion of sellers |
| Shipping limit | `max` + `min` + spread | Max = latest deadline (bottleneck); spread = coordination complexity |


In [7]:
# ── Aggregate items to order level ───────────────────────────────────────────
items_agg = (
    items_enriched
    .groupby('order_id', as_index=False)
    .agg(
        # ── Volume ───────────────────────────────────────────────────────────
        n_items                 = ('order_item_id',                   'max'),

        # ── Price & Freight ───────────────────────────────────────────────────
        total_price             = ('price',                           'sum'),
        max_item_price          = ('price',                           'max'),
        total_freight           = ('freight_value',                   'sum'),

        # ── Product physical dimensions ───────────────────────────────────────
        total_weight_g          = ('product_weight_g',                'sum'),
        max_item_weight_g       = ('product_weight_g',                'max'),
        total_volume_cm3        = ('product_volume_cm3',              'sum'),
        max_item_volume_cm3     = ('product_volume_cm3',              'max'),

        # ── Category ─────────────────────────────────────────────────────────
        # We do NOT select a single category (no mode).
        # n_unique_categories preserves all information in one number.
        # The dominant category will be identified after EDA.
        n_unique_categories     = ('product_category_name_english',   'nunique'),

        # ── Seller identity ───────────────────────────────────────────────────
        # We do NOT pick a single seller (no first).
        # Geographic seller features (distance, state) are built in NB04.
        n_unique_sellers        = ('seller_id',                       'nunique'),
        n_unique_seller_states  = ('seller_state',                    'nunique'),

        # ── Shipping deadlines ────────────────────────────────────────────────
        # max = latest deadline → the seller who is the bottleneck
        # min = earliest deadline → fastest seller in the order
        # spread = coordination complexity across sellers
        max_shipping_limit_date = ('shipping_limit_date',             'max'),
        min_shipping_limit_date = ('shipping_limit_date',             'min'),
    )
)

# Derived: shipping limit spread in days (complexity of multi-seller coordination)
items_agg['shipping_limit_spread_days'] = (
    items_agg['max_shipping_limit_date'] - items_agg['min_shipping_limit_date']
).dt.days

# Derived: flag for orders spanning more than one seller state
items_agg['has_multi_state_sellers'] = (
    items_agg['n_unique_seller_states'] > 1
).astype(int)

print(f'items_agg: {len(items_enriched):,} item rows → {len(items_agg):,} order rows')
print(f'Columns: {items_agg.shape[1]}')
print()
print(items_agg.dtypes.to_string())

items_agg: 112,650 item rows → 98,666 order rows
Columns: 16

order_id                              object
n_items                                int64
total_price                          float64
max_item_price                       float64
total_freight                        float64
total_weight_g                       float64
max_item_weight_g                    float64
total_volume_cm3                     float64
max_item_volume_cm3                  float64
n_unique_categories                    int64
n_unique_sellers                       int64
n_unique_seller_states                 int64
max_shipping_limit_date       datetime64[ns]
min_shipping_limit_date       datetime64[ns]
shipping_limit_spread_days             int64
has_multi_state_sellers                int32


In [8]:
# ── Sanity check: no new order IDs introduced ─────────────────────────────
orders_in_items = set(items_agg['order_id'])
orders_in_base  = set(orders['order_id'])

only_in_items  = orders_in_items - orders_in_base
only_in_orders = orders_in_base  - orders_in_items

print(f'Orders in items_agg not in orders table : {len(only_in_items)}  (expect 0)')
print(f'Orders in orders table not in items_agg : {len(only_in_orders)}')
print(f'  → These orders have no item records and will get NaN after LEFT join (handled below)')

Orders in items_agg not in orders table : 2211  (expect 0)
Orders in orders table not in items_agg : 0
  → These orders have no item records and will get NaN after LEFT join (handled below)


---
## Aggregate Payments to Order Level

Multiple payment rows per order arise when customers split payment across methods.
We aggregate to one row per order.

**Note on `payment_type`:** `mode()` is acceptable here because payment method is a
single-dimension categorical (it describes how the customer chose to pay, not which
 of several independent entities is more important). A customer who pays 90% by credit card
 and 10% by voucher is correctly described as a "credit card" customer.

In [9]:
# ── Aggregate payments to order level ────────────────────────────────────────
payments_agg = (
    payments
    .groupby('order_id', as_index=False)
    .agg(
        total_payment_value = ('payment_value',        'sum'),
        n_payment_methods   = ('payment_sequential',   'max'),
        max_installments    = ('payment_installments', 'max'),
        payment_type        = ('payment_type',
                               lambda x: x.mode()[0] if not x.mode().empty else 'unknown')
    )
)

print(f'Payments aggregated: {len(payments):,} rows → {len(payments_agg):,} orders')

Payments aggregated: 103,886 rows → 99,440 orders


---
## Deduplicate Reviews

**What NB01 confirmed:**
- `reviews_clean.csv` contains **99,224 rows**
- Full duplicate rows: **0** - no identical rows exist
- `order_id` is not unique: there are **551 extra rows** where the same `order_id`
  appears more than once - meaning some customers submitted multiple reviews

**Important distinction:**
These are *not* full-row duplicates (that is why NB01 duplicate audit reported 0 for reviews).
Each extra row has a different `review_id`, score, or timestamp - they are separate
submissions by the same customer for the same order.

**Decision:** keep the chronologically first review per `order_id` (by `review_creation_date`).
After dedup: **98,673 unique orders** retain a review score. **551 extra rows** are removed.

> ⚠️ **Leakage reminder:** `review_score` is recorded after delivery - it must not be used
> as a model feature. Kept here for EDA. Dropped before modeling in NB04.


In [10]:
# ── Deduplicate reviews — keep first review per order ────────────────────────
# NB01 context:
#   reviews_clean.csv  : 99,224 rows
#   Full duplicate rows : 0  (no identical rows — confirmed in NB01 audit)
#   Duplicate order_ids : 551 extra rows where order_id appears more than once
#
# These are NOT full-row duplicates — each row has a different review_id,
# score, or timestamp. Strategy: keep chronologically first review per order.

n_before        = len(reviews)
n_dup_rows      = reviews['order_id'].duplicated().sum()
n_affected_orders = reviews[reviews['order_id'].duplicated(keep=False)]['order_id'].nunique()

reviews_slim = (
    reviews
    .sort_values('review_creation_date')
    .drop_duplicates(subset='order_id', keep='first')
    [['order_id', 'review_score']]
)

n_after = len(reviews_slim)

print(f'reviews_clean.csv          : {n_before:,} rows')
print(f'Duplicate order_id rows    : {n_dup_rows:,}  (extra rows, not full-row duplicates)')
print(f'Orders with >1 review      : {n_affected_orders:,}  (unique orders affected)')
print(f'After dedup                : {n_after:,} unique orders')
print(f'Rows removed               : {n_before - n_after:,}')


reviews_clean.csv          : 99,224 rows
Duplicate order_id rows    : 551  (extra rows, not full-row duplicates)
Orders with >1 review      : 547  (unique orders affected)
After dedup                : 98,673 unique orders
Rows removed               : 551


---
## Build Final Merged Dataset

We anchor all joins to the `orders` table using LEFT JOINs.
This guarantees row count is preserved — every delivered order gets one row.

### Join map

```
orders  (96,455 rows — anchor)
  ├── customers        on customer_id
  ├── items_agg        on order_id
  ├── payments_agg     on order_id
  ├── reviews_slim     on order_id
  └── customer_geo     on customer_zip_code_prefix

NOT joined here:
  └── seller_geo  ← deferred to NB04 (requires choosing a seller per order)
```

In [11]:
# ── Merge all tables ──────────────────────────────────────────────────────────
df = orders.copy()

# 1. Customer demographics
df = df.merge(
    customers[['customer_id', 'customer_unique_id',
               'customer_zip_code_prefix', 'customer_state', 'customer_city']],
    on='customer_id', how='left'
)

# 2. Item-level aggregates (physics, price, seller count, deadlines)
df = df.merge(items_agg,    on='order_id', how='left')

# 3. Payment aggregates
df = df.merge(payments_agg, on='order_id', how='left')

# 4. Review score (EDA-only — not for modeling)
df = df.merge(reviews_slim, on='order_id', how='left')

# 5. Customer geolocation (one zip → one centroid, safe 1:1 join)
df = df.merge(customer_geo, on='customer_zip_code_prefix', how='left')

print(f'Final shape       : {df.shape}')
print(f'Row count check   : {len(df):,} orders (anchor = {len(orders):,}) — match: {len(df) == len(orders)}')
print(f'Duplicate orders  : {df["order_id"].duplicated().sum()}')
print(f'Late rate         : {df["is_late"].mean()*100:.2f}%  (must match NB01: 8.11%)')

Final shape       : (96455, 39)
Row count check   : 96,455 orders (anchor = 96,455) — match: True
Duplicate orders  : 0
Late rate         : 8.11%  (must match NB01: 8.11%)


---
## Quality Checks

In [12]:
# ── Missing value audit ───────────────────────────────────────────────────────
print('Missing values (non-zero only):')
missing = df.isnull().sum()
missing_nz = missing[missing > 0].sort_values(ascending=False)
print(missing_nz.to_string())

print()
print('Expected missing values and reasons:')
expected = {
    'review_score'              : 'Not all customers submit a review (0.7%) — captured by has_review',
    'customer_lat/lng'          : 'Zip code not found in geolocation table (0.27%)',
    'max/min_shipping_limit_date': 'Orders with no item records (edge case)',
    'n_items, total_price ...'  : 'Same — orders with no item records',
}
for col, reason in expected.items():
    print(f'  {col:<35}: {reason}')

Missing values (non-zero only):
review_score           646
customer_lat           264
customer_lng           264
max_item_weight_g       16
max_item_volume_cm3     16
total_payment_value      1
n_payment_methods        1
max_installments         1
payment_type             1

Expected missing values and reasons:
  review_score                       : Not all customers submit a review (0.7%) — captured by has_review
  customer_lat/lng                   : Zip code not found in geolocation table (0.27%)
  max/min_shipping_limit_date        : Orders with no item records (edge case)
  n_items, total_price ...           : Same — orders with no item records


In [13]:
# ── Shape and target integrity ───────────────────────────────────────────────
assert len(df) == len(orders),         f'Row count mismatch: {len(df)} vs {len(orders)}'
assert df['order_id'].nunique() == len(df), 'Duplicate order_ids found'
assert abs(df['is_late'].mean() - 0.0811) < 0.001, 'Late rate changed — check joins'

print('All assertions passed.')
print(f'  Rows          : {len(df):,}')
print(f'  Unique orders : {df["order_id"].nunique():,}')
print(f'  Columns       : {df.shape[1]}')
print(f'  Late rate     : {df["is_late"].mean()*100:.2f}%')

All assertions passed.
  Rows          : 96,455
  Unique orders : 96,455
  Columns       : 39
  Late rate     : 8.11%


---
## Handle Remaining Missing Values

| Column | Action | Reason |
|--------|--------|--------|
| `review_score` | Keep NaN + add `has_review` flag | Absence of review is informative |
| `customer_lat/lng` | Keep NaN | `customer_state` available as fallback geographic feature |
| Item columns (weight, price…) | Fill with median | Very few orders (~16) have no item records |
| Payment columns | Fill with 0 / 'unknown' | Very few orders (~1) have no payment records |
| `shipping_limit_spread_days` | Fill with 0 | Single-item orders have no spread by definition |

In [14]:
# ── Flag: has_review ─────────────────────────────────────────────────────────
df['has_review'] = df['review_score'].notna().astype(int)

# ── Item columns (orders with no item record) ─────────────────────────────────
item_numeric_cols = [
    'n_items', 'total_price', 'max_item_price',
    'total_freight', 'total_weight_g', 'max_item_weight_g',
    'total_volume_cm3', 'max_item_volume_cm3',
    'n_unique_categories', 'n_unique_sellers', 'n_unique_seller_states',
]
for col in item_numeric_cols:
    if col in df.columns and df[col].isnull().any():
        median_val = df[col].median()
        n_filled   = df[col].isnull().sum()
        df[col]    = df[col].fillna(median_val)
        print(f'  {col:<35}: filled {n_filled} NaN with median ({median_val:.1f})')

# ── Shipping limit spread ─────────────────────────────────────────────────────
if df['shipping_limit_spread_days'].isnull().any():
    n = df['shipping_limit_spread_days'].isnull().sum()
    df['shipping_limit_spread_days'] = df['shipping_limit_spread_days'].fillna(0)
    print(f'  {"shipping_limit_spread_days":<35}: filled {n} NaN with 0')

df['has_multi_state_sellers'] = df['has_multi_state_sellers'].fillna(0).astype(int)

# ── Payment columns ───────────────────────────────────────────────────────────
df['n_payment_methods']   = df['n_payment_methods'].fillna(0).astype(int)
df['max_installments']    = df['max_installments'].fillna(0).astype(int)
df['total_payment_value'] = df['total_payment_value'].fillna(0)
df['payment_type']        = df['payment_type'].fillna('unknown')

print()
print('Remaining NaN after fill (expected — geographic and review only):')
remaining = df.isnull().sum()
print(remaining[remaining > 0].to_string())

  max_item_weight_g                  : filled 16 NaN with median (700.0)
  max_item_volume_cm3                : filled 16 NaN with median (6468.0)

Remaining NaN after fill (expected — geographic and review only):
review_score    646
customer_lat    264
customer_lng    264


---
## Drop Zero-Variance Columns

Before saving, we remove columns that carry no information:

| Column | Why dropped |
|--------|-------------|
| `order_status` | Always = 'delivered' after scope filter — same reason |

 These are not 'dropped before modeling' - they are dropped here because they add noise
 to the schema with zero informational value at any stage.

In [15]:
# ── Drop zero-variance columns ────────────────────────────────────────────────
# is_delivered : always 1 (scope filter in NB01 kept only delivered orders)
# order_status : always 'delivered' (same reason)
# Confirmed before drop:
print('Pre-drop verification:')
print(f'  is_delivered unique values : {df["is_delivered"].unique()}  — always 1')
print(f'  order_status unique values : {df["order_status"].unique()}  — always delivered')

cols_to_drop = ['is_delivered', 'order_status']
df = df.drop(columns=cols_to_drop)

print(f'\nDropped: {cols_to_drop}')
print(f'Shape after drop: {df.shape}')


Pre-drop verification:
  is_delivered unique values : [1]  — always 1
  order_status unique values : ['delivered']  — always delivered

Dropped: ['is_delivered', 'order_status']
Shape after drop: (96455, 38)


---
##  Save Outputs

In [16]:
# ── Save items_enriched (item-level reference) ───────────────────────────────
import os
os.makedirs('../data/processed/', exist_ok=True)

items_enriched.to_csv('../data/processed/items_enriched.csv', index=False)
print(f'Saved: items_enriched.csv  → {items_enriched.shape}  (item-level reference for NB03 & NB04)')

# ── Save olist_merged (order-level main dataset) ──────────────────────────────
df.to_csv('../data/processed/olist_merged.csv', index=False)
print(f'Saved: olist_merged.csv    → {df.shape}  (order-level dataset for EDA & modeling)')

Saved: items_enriched.csv  → (112650, 22)  (item-level reference for NB03 & NB04)
Saved: olist_merged.csv    → (96455, 38)  (order-level dataset for EDA & modeling)


---

### Quick Data Inspection

In [17]:
def quick_inspect(name, dataframe):
    print(f'\n=== {name} ===')
    print(f'Shape: {dataframe.shape}')
    print('\nInfo:')
    display(dataframe.info())
    print('\nHead:')
    display(dataframe.head())

# Apply
quick_inspect('items_enriched', items_enriched)
quick_inspect('olist_merged', df)


=== items_enriched ===
Shape: (112650, 22)

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 22 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   order_id                       112650 non-null  object        
 1   order_item_id                  112650 non-null  int64         
 2   product_id                     112650 non-null  object        
 3   seller_id                      112650 non-null  object        
 4   shipping_limit_date            112650 non-null  datetime64[ns]
 5   price                          112650 non-null  float64       
 6   freight_value                  112650 non-null  float64       
 7   is_high_value                  112650 non-null  int64         
 8   is_zero_freight                112650 non-null  int64         
 9   product_category_name          112632 non-null  object        
 10  product_name_leng

None


Head:


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,is_high_value,is_zero_freight,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,seller_zip_code_prefix,seller_state,seller_city,product_volume_cm3
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,0,0,cool_stuff,58.00,598.00,4.00,650.00,28.00,9.00,14.00,cool_stuff,27277,SP,volta redonda,3528.00
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,0,0,pet_shop,56.00,239.00,2.00,30000.00,50.00,30.00,40.00,pet_shop,3471,SP,sao paulo,60000.00
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,0,0,moveis_decoracao,59.00,695.00,2.00,3050.00,33.00,13.00,33.00,furniture_decor,37564,MG,borda da mata,14157.00
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,0,0,perfumaria,42.00,480.00,1.00,200.00,16.00,10.00,15.00,perfumery,14403,SP,franca,2400.00
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,0,0,ferramentas_jardim,59.00,409.00,1.00,3750.00,35.00,40.00,30.00,garden_tools,87900,PR,loanda,42000.00



=== olist_merged ===
Shape: (96455, 38)

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 96455 entries, 0 to 96454
Data columns (total 38 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       96455 non-null  object        
 1   customer_id                    96455 non-null  object        
 2   order_purchase_timestamp       96455 non-null  datetime64[ns]
 3   order_approved_at              96455 non-null  datetime64[ns]
 4   order_delivered_carrier_date   96455 non-null  datetime64[ns]
 5   order_delivered_customer_date  96455 non-null  datetime64[ns]
 6   order_estimated_delivery_date  96455 non-null  datetime64[ns]
 7   is_late                        96455 non-null  int64         
 8   delivery_status                96455 non-null  object        
 9   delivery_days                  96455 non-null  int64         
 10  has_timestamp_issue            964

None


Head:


,order_id,customer_id,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,is_late,delivery_status,delivery_days,has_timestamp_issue,customer_unique_id,customer_zip_code_prefix,customer_state,customer_city,n_items,total_price,max_item_price,total_freight,total_weight_g,max_item_weight_g,total_volume_cm3,max_item_volume_cm3,n_unique_categories,n_unique_sellers,n_unique_seller_states,max_shipping_limit_date,min_shipping_limit_date,shipping_limit_spread_days,has_multi_state_sellers,total_payment_value,n_payment_methods,max_installments,payment_type,review_score,customer_lat,customer_lng,has_review
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,0,early,8,0,7c396fd4830fd04220f754e42b4e5bff,3149,SP,sao paulo,1,29.99,29.99,8.72,500.00,500.00,1976.00,1976.00,1,1,1,2017-10-06 11:07:15,2017-10-06 11:07:15,0,0,38.71,3,1,voucher,4.00,-23.58,-46.59,1
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,0,early,13,0,af07308b275d755c9edb36a90c618231,47813,BA,barreiras,1,118.70,118.70,22.76,400.00,400.00,4693.00,4693.00,1,1,1,2018-07-30 03:24:27,2018-07-30 03:24:27,0,0,141.46,1,1,boleto,4.00,-12.18,-44.66,1
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,0,early,9,0,3a653a41f6f9fc3d2a113cf8398680e8,75265,GO,vianopolis,1,159.90,159.90,19.22,420.00,420.00,9576.00,9576.00,1,1,1,2018-08-13 08:55:23,2018-08-13 08:55:23,0,0,179.12,1,3,credit_card,5.00,-16.75,-48.51,1
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,0,early,13,0,7c142cf63193a1473d2e66489a9ae977,59296,RN,sao goncalo do amarante,1,45.00,45.00,27.20,450.00,450.00,6000.00,6000.00,1,1,1,2017-11-23 19:45:59,2017-11-23 19:45:59,0,0,72.20,1,1,credit_card,5.00,-5.77,-35.27,1
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,0,early,2,0,72632f0f9dd73dfee390c9b22eb56dd6,9195,SP,santo andre,1,19.90,19.90,8.72,250.00,250.00,11475.00,11475.00,1,1,1,2018-02-19 20:31:37,2018-02-19 20:31:37,0,0,28.62,1,1,credit_card,5.00,-23.68,-46.51,1


---
## Summary

### What was built

| Output | Rows | Columns | Granularity |
|--------|------|---------|-------------|
| `items_enriched.csv` | 112,650 | 22 | One row = one item |
| `olist_merged.csv`   | 96,455  | 38 | One row = one order |

### Aggregation decisions

| Feature | Decision | Deferred to |
|---------|----------|-------------|
| `product_category` | Not selected - `n_unique_categories` instead | NB04 after EDA |
| `seller_state` | Not selected - `n_unique_seller_states` + `has_multi_state_sellers` instead | NB04 after EDA |
| `seller_lat/lng` | Not joined - requires choosing a single seller | NB04 after EDA |
| `max_shipping_limit_date` | Added - identifies the bottleneck deadline | Used directly in NB03/NB04 |
| `shipping_limit_spread_days` | Added - measures multi-seller coordination complexity | Used directly in NB03/NB04 |

### Columns deferred / EDA-only

| Column | Reason | Action |
|--------|--------|--------|
| `delivery_days` | Computed from actual delivery - post-delivery leakage | EDA only, exclude before modeling |
| `delivery_status` | Same - derived from actual delivery date | EDA only, exclude before modeling |
| `review_score` | Collected after delivery | EDA only, exclude before modeling |

**Next step:** `03_eda.ipynb` - use `olist_merged.csv` for order-level analysis and
 `items_enriched.csv` for seller and category deep-dives.

---
## Final Dataset Schema - olist_merged.csv

| Column | Source | Type | Notes |
|--------|--------|------|-------|
| `order_id` | orders | string | Primary key |
| `customer_id` | orders | string | Per-order customer key |
| `order_purchase_timestamp` | orders | datetime | Purchase time |
| `order_approved_at` | orders | datetime | Payment approval |
| `order_delivered_carrier_date` | orders | datetime | Carrier handoff |
| `order_delivered_customer_date` | orders | datetime | Actual delivery |
| `order_estimated_delivery_date` | orders | datetime | Promised delivery date |
| **`is_late`** | orders | int 0/1 | **Target variable** |
| `delivery_status` | orders | string | 'early'/'late' — EDA only |
| `delivery_days` | orders | int | Days purchase→delivery — EDA only |
| `has_timestamp_issue` | orders | int 0/1 | Suspicious timestamp ordering |
| `customer_unique_id` | customers | string | True customer identifier |
| `customer_zip_code_prefix` | customers | int | Zip prefix |
| `customer_state` | customers | string | State abbreviation |
| `customer_city` | customers | string | Customer city |
| `n_items` | items | int | Number of items in order |
| `total_price` | items | float | Sum of all item prices |
| `max_item_price` | items | float | Most expensive item |
| `total_freight` | items | float | Sum of all freight costs |
| `total_weight_g` | items | float | Total order weight |
| `max_item_weight_g` | items | float | Heaviest single item |
| `total_volume_cm3` | items | float | Total order volume |
| `max_item_volume_cm3` | items | float | Largest single item volume |
| `n_unique_categories` | items | int | Number of distinct product categories |
| `n_unique_sellers` | items | int | Number of distinct sellers |
| `n_unique_seller_states` | items | int | Geographic spread of sellers |
| `has_multi_state_sellers` | items | int 0/1 | Order spans multiple seller states |
| `max_shipping_limit_date` | items | datetime | Latest seller deadline (bottleneck) |
| `min_shipping_limit_date` | items | datetime | Earliest seller deadline |
| `shipping_limit_spread_days` | items | int | Days between first and last deadline |
| `total_payment_value` | payments | float | Total amount paid |
| `n_payment_methods` | payments | int | Number of payment methods used |
| `max_installments` | payments | int | Maximum installments |
| `payment_type` | payments | string | Most common payment method |
| `review_score` | reviews | float | 1–5 rating — EDA only, post-delivery leakage |
| `has_review` | reviews | int 0/1 | 1 = customer submitted a review |
| `customer_lat` | geolocation | float | Customer latitude (centroid) |
| `customer_lng` | geolocation | float | Customer longitude (centroid) |